In [1]:
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

# Tải mô hình VGG16 đã được huấn luyện sẵn trên ImageNet 
conv_base = keras.applications.vgg16.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(180, 180, 3))
# conv_base.summary()  # In ra cấu trúc mô hình 

# Đường dẫn tới dữ liệu huấn luyện, validation và test
train_dir = "cats_vs_dogs_small/train"
validation_dir = "cats_vs_dogs_small/validation"
test_dir = "cats_vs_dogs_small/test"

# Tạo dataset huấn luyện từ thư mục, resize ảnh về (180, 180), batch_size = 32
train_dataset = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(180, 180),
    batch_size=32
)

# Tạo dataset validation
validation_dataset = keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(180, 180),
    batch_size=32
)

# Tạo dataset test
test_dataset = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(180, 180),
    batch_size=32
)

2025-09-25 10:08:28.569278: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-09-25 10:08:28.569320: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-09-25 10:08:28.569334: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-09-25 10:08:28.569385: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-25 10:08:28.569410: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [2]:
# Hàm trích xuất đặc trưng (feature) từ conv_base
def get_features_and_labels(dataset):
    all_features = []
    all_labels = []
    for images, labels in dataset:
        # Chuẩn hóa dữ liệu theo chuẩn của VGG16
        preprocessed_images = keras.applications.vgg16.preprocess_input(images)
        # Dự đoán đặc trưng bằng conv_base
        features = conv_base.predict(preprocessed_images)
        all_features.append(features)
        all_labels.append(labels)
    # Nối tất cả đặc trưng và nhãn lại thành mảng lớn
    return np.concatenate(all_features), np.concatenate(all_labels)

# Trích xuất đặc trưng cho train, validation, test
train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)

train_features.shape  # Kiểm tra shape của đặc trưng

# HUẤN LUYỆN KHÔNG CÓ DATA AUGMENTATION 

# Xây dựng mô hình phân loại (dựa trên feature từ conv_base)
inputs = keras.Input(shape=(5, 5, 512))   
x = layers.Flatten()(inputs)              
x = layers.Dense(256)(x)                  
x = layers.Dropout(0.5)(x)                
outputs = layers.Dense(1, activation="sigmoid")(x)  
model = keras.Model(inputs, outputs)

# Compile mô hình
model.compile(loss="binary_crossentropy",
        optimizer="rmsprop",
        metrics=["accuracy"])

# Callback lưu lại mô hình tốt nhất dựa trên val_loss
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction.h5", # check lại filepath nếu lỗi
        save_best_only=True,
        monitor="val_loss")
]

# Huấn luyện mô hình
history = model.fit(
    train_features, train_labels,
    epochs=20,
    validation_data=(val_features, val_labels),
    callbacks=callbacks)

# Vẽ biểu đồ accuracy và loss
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)

plt.plot(epochs, acc, "bo", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()

plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

# HUẤN LUYỆN VỚI DATA AUGMENTATION 

# Khởi tạo lại conv_base
conv_base = keras.applications.vgg16.VGG16(
    weights="imagenet",
    include_top=False)
conv_base.trainable = False  # Ban đầu không train lại conv_base

# In số lượng tham số trainable trước và sau khi freeze conv_base
conv_base.trainable = True
print("This is the number of trainable weights "
    "before freezing the conv base:", len(conv_base.trainable_weights))
conv_base.trainable = False
print("This is the number of trainable weights "
    "after freezing the conv base:", len(conv_base.trainable_weights))

# Tạo data augmentation (lật ngang, xoay, zoom ngẫu nhiên)
data_augmentation = keras.Sequential(
    [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    ]
)

# Xây dựng mô hình với augmentation + conv_base
inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)                  
x = keras.applications.vgg16.preprocess_input(x) 
x = conv_base(x)                               
x = layers.Flatten()(x)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(loss="binary_crossentropy",
        optimizer="rmsprop",
        metrics=["accuracy"])

# Callback lưu mô hình tốt nhất
callbacks = [
    keras.callbacks.ModelCheckpoint(
    filepath="feature_extraction_with_data_augmentation.h5", # check lại filepath nếu lỗi
    save_best_only=True,
    monitor="val_loss")
]

# Huấn luyện mô hình trực tiếp trên ảnh (không trích feature sẵn)
history = model.fit(
    train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=callbacks)

# Vẽ biểu đồ accuracy và loss
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)

plt.plot(epochs, acc, "bo", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()

plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

# Đánh giá mô hình tốt nhất trên tập test
test_model = keras.models.load_model(
    "feature_extraction_with_data_augmentation.h5")
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")


2025-09-25 10:08:57.058898: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 [==============================] - 0s 42ms/step


KeyboardInterrupt: 